In [1]:
from langchain.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage


In [16]:
from langchain.chat_models import init_chat_model
model=init_chat_model(
    "gemini-3.5-flash",
    model_provider="google_genai",
)
@tool

def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"


model_with_tools=model.bind_tools([get_weather])


In [17]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content=[] additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "Boston"}'}, '__gemini_function_call_thought_signatures__': {'call_242903': 'Eu0CCuoCAWkUfROnZ89dq52mjmjzW5RwX+0lX8MtovWoKwS2a39bFIzXrzBdY0g8sxTqxE5DnrrWYHVsupP1vcpEma/l0LKzE+fOlLndni1HZjLV4fkUROwJxlxvhtw3O0f4hQcfL5BMY3nSgAHZst0xEyEW7MhgWVAqukgeh8oheFS3XvkEs+ANf9cK8eqqOLCjx8cUicjuy6Jwce9rwkK09k9v6Yi06W4ksJSk3UFCdwKSz4g0nAX294fzgz70uqPejVsx0pQCY/Ht1O9JQX7Pp2/pqETrEbaanqxA7g8HUg9POQ24BCqhyCwRS1j1+pM0g4Gyt8IOrNA4QxONS5WJt0i/2U1W71Xft5yxqP9vOGhy3aXSvsfX62T8HV7lVH2Vs71Ip9JJhiOY3Po8a8SefcMxPrV9cTJhGTKOs4ghEpx0b7b1FghGdn+9p3EPjFQsS94szxfEGUNuib9Y8v7dRlXPO3tdKs10Wfgm7Ag='}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.6-flash', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--01a18b22-ddd6-73cb-ba69-9ee2d6017fcf-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'call_242903', 'type': 'tool_call'}] invalid_tool_calls=[] usage_meta

TOOL EXECUTION LOOPS

In [20]:
mes=[{"role": "user", "content": "What is the capital of France?"}]
ai_msg=model_with_tools.invoke(mes)

In [21]:
mes.append(ai_msg)

In [22]:
for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    mes.append(tool_result)

In [25]:
mes = [HumanMessage(content="What's the weather like in Boston?")]

ai_msg = model_with_tools.invoke(mes)
mes.append(ai_msg)

for tool_call in ai_msg.tool_calls:
    result = get_weather.invoke(tool_call)
    mes.append(
        ToolMessage(
            content=result,
            tool_call_id=tool_call["id"],
        )
    )

final_response = model_with_tools.invoke(mes)
print(final_response.text)

It's currently sunny in Boston!
